In [ ]:
# Rutas portables: ejecutar desde notebooks/ o desde la raíz clonada.
from pathlib import Path
import sys, os
_repo = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs/pipeline.json').exists())
sys.path.insert(0, str(_repo / 'src'))
from pipeline_runtime import prepare_runtime
_runtime = prepare_runtime(_repo)
os.chdir(_runtime)
sys.path.insert(0, str(_runtime))

import json
_central = json.loads((_repo / 'configs/pipeline.json').read_text(encoding='utf-8'))

from segmentation_validation import require_generated_partition
require_generated_partition(_repo)


# 1.CARGA LIBRERIA

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import colormaps
from collections import defaultdict
from sklearn.metrics import (roc_auc_score, roc_curve, classification_report, confusion_matrix)
import statsmodels.api as sm
import shap
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import time
import os

# 2.CARGA DE DATOS

In [ ]:
df = pd.read_csv("2.OUTPUT/1.1.0.TAB_SEGM_TRAIN_TEST_OOT.csv")
df.head(5)

In [ ]:
df.shape

In [ ]:
df['SEGMENTO'].unique()

# 3.PARTICION DE LA MUESTRA

In [ ]:
df_train = df[df['SEGMENTO'] == 'TRAIN'].copy()
y_train = df[df['SEGMENTO'] == 'TRAIN']['indicador_default'].copy()

df_test = df[df['SEGMENTO'] == 'TEST'].copy()
y_test = df[df['SEGMENTO'] == 'TEST']['indicador_default'].copy()

df_oot = df[df['SEGMENTO'] == 'OOT'].copy()
y_oot = df[df['SEGMENTO'] == 'OOT']['indicador_default'].copy()

In [ ]:
vars_excluir = ['periodo_observacion', 'cliente_id','indicador_default','SEGMENTO']

In [ ]:
X_train = df_train.drop(columns=vars_excluir)
X_test = df_test.drop(columns=vars_excluir)
X_oot = df_oot.drop(columns=vars_excluir)

# 4.ANALISIS UNIVARIADO

In [ ]:
# Inicializar contenedores
reporte_variabilidad = []
variables_descartadas = []

# Evaluamos cada columna de X_train
for col in X_train.columns:
    total_obs = len(X_train)
    col_type = X_train[col].dtype

    # Variables categoricas
    if (pd.api.types.is_object_dtype(col_type) or isinstance(col_type, pd.CategoricalDtype)
        or pd.api.types.is_string_dtype(col_type)):

        top_freq = X_train[col].value_counts().iloc[0]
        pct_concentracion = (top_freq / total_obs) * 100

        if pct_concentracion > _central['univariate']['variability_max_pct']:
            estado = "Rechazado"
            motivo = f"Baja variabilidad categorica: Moda concentra el {pct_concentracion:.2f}%"
            variables_descartadas.append(col)
        else:
            estado = "Aprobado"
            motivo = f"Variabilidad Ok. Mayor categoria concentra el {pct_concentracion:.2f}%"

    # Caso 2 variables numericas
    else:
        datos_num = X_train[col].dropna()

        if len(datos_num) == 0:
            estado = "Rechazado"
            motivo = "Variable completamente vacia en train"
            variables_descartadas.append(col)
        else:
            conteo_moda = datos_num.value_counts().iloc[0]
            valor_moda = datos_num.value_counts().index[0]

            pct_concentracion = (conteo_moda / total_obs) * 100

            if pct_concentracion > _central['univariate']['variability_max_pct']:
                estado = "Rechazado"
                motivo = f"Baja variabilidad numerica: el valor ({valor_moda}) concentra el {pct_concentracion:.2f}%"
                variables_descartadas.append(col)
            else:
                estado = "Aprobado"
                motivo = f"Variabilidad numerica OK: Mayor concentracion es de {pct_concentracion:.2f}%"

    reporte_variabilidad.append({
        'Paso / Filtro': 'Condicion 1: Variabilidad',
        'Variable': col,
        'Tipo': 'Numerica' if pd.api.types.is_numeric_dtype(col_type) else 'Categorica',
        'Estado': estado,
        'Motivo / Sustento': motivo
    })

df_reporte_univariado = pd.DataFrame(reporte_variabilidad)

# Obtenemos
X_train_sin_bv = X_train.drop(columns=variables_descartadas, errors='ignore')
X_test_sin_bv = X_test.drop(columns=variables_descartadas, errors='ignore')
X_oot_sin_bv = X_oot.drop(columns=variables_descartadas, errors='ignore') if 'X_oot' in locals() else None

print("=" * 80)
print("Filtro univariado completado (Umbral >85%)")
print(f"Variables iniciales : {X_train.shape[1]}")
print(f"Variables evaluadas : {len(reporte_variabilidad)}")
print(f"Variables aprobadas : {X_train_sin_bv.shape[1]}")
print(f"Total de variables descartadas: {len(variables_descartadas)}")
print("=" * 80)
print("Variables descartadas:", variables_descartadas)
print("=" * 80)

In [ ]:
import math
vars_desc_num = [
    v for v in variables_descartadas
    if pd.api.types.is_numeric_dtype(X_train[v])
]

n_cols = 4
n_rows = math.ceil(len(vars_desc_num) / n_cols)

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(24, n_rows * 4)
)

axes = axes.flatten()

for i, var in enumerate(vars_desc_num):

    # Top frecuencias TRAIN
    freq_train = (
        X_train[var]
        .value_counts(normalize=True, dropna=False)
        .head(10)
        * 100
    )

    ax = axes[i]

    freq_train.plot(
        kind="bar",
        color="steelblue",
        alpha=0.8,
        ax=ax
    )

    ax.axhline(
        y=85,
        color="red",
        linestyle="--",
        linewidth=2,
        label="Umbral 85%"
    )

    ax.set_title(var, fontsize=9)
    ax.set_ylabel("%")
    ax.tick_params(axis="x", labelrotation=45)
    ax.legend(fontsize=7)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# ANALISIS UNIVARIADO -- CONDICION 4: (NULOS > 40%)
# ============================================================

features_v2 = X_train_sin_bv.columns.to_list()

variables_descartadas_f3 = []
reporte_f3 = []

for col in features_v2:

    total_obs = len(X_train_sin_bv)
    nulos = X_train_sin_bv[col].isna().sum()
    pct_nulos = (nulos / total_obs) * 100

    if pct_nulos > _central['univariate']['missing_max_pct']:
        estado = "Rechazado"
        motivo = f"Condicion 4: Porcentaje de nulos ({pct_nulos:.2f}% > 40%)"
        variables_descartadas_f3.append(col)

    else:
        estado = "Aprobado"
        motivo = f"Porcentaje de nulos OK ({pct_nulos:.2f}%)"

    reporte_f3.append({
        'Paso / filtro': 'Condicion 4: Nulos',
        'Variable': col,
        'Tipo': 'numerica' if pd.api.types.is_numeric_dtype(X_train_sin_bv[col]) else 'categorica',
        'Estado': estado,
        'Motivo/Sustento': motivo
    })

# Reporte
df_reporte_f3 = pd.DataFrame(reporte_f3)

# Consolidar reporte general
df_reporte_univariado = pd.concat(
    [pd.DataFrame(reporte_variabilidad), df_reporte_f3],
    ignore_index=True
)

# Eliminar variables rechazadas
X_train_v3 = X_train_sin_bv.drop(columns=variables_descartadas_f3, errors='ignore')
X_test_v3 = X_test_sin_bv.drop(columns=variables_descartadas_f3, errors='ignore')
X_oot_v3 = (
    X_oot_sin_bv.drop(columns=variables_descartadas_f3, errors='ignore')
    if X_oot_sin_bv is not None
    else None
)

print("=" * 80)
print("Filtro Condicion 4: Nulos > 40%")
print(f" - Variables analizadas : {X_train_sin_bv.shape[1]}")
print(f" - Variables rechazadas : {len(variables_descartadas_f3)}")
print(f" - Variables aprobadas  : {X_train_v3.shape[1]}")

if variables_descartadas_f3:
    print(f" - Lista de variables descartadas: {variables_descartadas_f3}")

print("=" * 80)

In [ ]:
reporte_nulos = []

for var in variables_descartadas_f3:

    nulos = X_train_sin_bv[var].isna().sum()
    pct_nulos = round(nulos / len(X_train_sin_bv) * 100, 2)

    reporte_nulos.append({
        "Variable": var,
        "Razón Eliminación": "Exceso de nulos",
        "Nulos": nulos,
        "% Nulos": pct_nulos,
        "Umbral": "40%"
    })

df_reporte_nulos = (
    pd.DataFrame(reporte_nulos)
    .sort_values("% Nulos", ascending=False)
)

df_reporte_nulos.head(18)

In [ ]:

os.makedirs("4.ANALIS_UV_BV_MV", exist_ok=True)

df_reporte_nulos.to_csv(
    "4.ANALIS_UV_BV_MV/4.2.TAB_CCS_BCS1_A_UNIV_NULOS.csv",
    index=False
)

# PSI temporal mensual ? gu?a de ejecuci?n

Ejecute PSI.1?PSI.6 para revisar el c?lculo y PSI.7?PSI.9 para visualizarlo. PSI.8 es el dashboard principal. PSI.10 consolida el reporte, PSI.11 construye las matrices y **PSI.12 es la ?nica secci?n PSI que escribe resultados en disco**. Puede omitir PSI.12 durante la exploraci?n. Cambiar una variable, filtro o gr?fico no requiere recalcular PSI.3.

| Archivo | Responsabilidad y uso |
| --- | --- |
| `psi_temporal.py` | M?dulo importado. `learn_bins`, `transform`, `calculate_psi` y `classify` implementan la metodolog?a; `run_analysis` los coordina. No se ejecuta directamente. |
| `test_psi_temporal.py` | Pruebas de metodolog?a y de las interfaces. Ejec?telas cuando modifique el m?dulo; no ejecutan ni escriben el notebook. |
| `ejecutar_psi_notebook.py` | Automatiza las celdas mediante Python y captura tablas y figuras. No es necesario para trabajo interactivo. Por defecto guarda una copia ejecutada; `--in-place` actualiza este notebook. |
| `informe_psi.py` | Genera README desde los archivos exportados por PSI.12; no calcula PSI. |
| Este notebook | Ejecute sus celdas secuencialmente desde la carga de datos. |

Desde la ra?z del proyecto, en Windows PowerShell:
```powershell
.\.venv\Scripts\python.exe -m unittest test_psi_temporal -v
.\.venv\Scripts\python.exe ejecutar_psi_notebook.py
.\.venv\Scripts\python.exe informe_psi.py
```

Los meses de referencia no son evaluaciones temporalmente independientes. PSI mide cambio distributivo, no capacidad predictiva. Los filtros anteriores >85% de concentraci?n y >40% de nulos se mantienen intactos. TEST y OOT son diagn?sticos: las clasificaciones dependen exclusivamente de TRAIN.


## PSI.1 ? Importaci?n y configuraci?n

**Objetivo:** Importar el m?dulo y validar expl?citamente los par?metros actuales.

**Procedencia:** `psi_temporal.py`; el notebook organiza las llamadas y muestra sus resultados.

**Resultado esperado:** Tabla de configuraci?n; ninguna exportaci?n ni c?lculo PSI.

`importlib.reload(psi)` vuelve a leer el archivo del m?dulo y restablece sus par?metros. ?selo solo si edit? `psi_temporal.py` en una sesi?n abierta; despu?s vuelva a ejecutar PSI.1. No se recarga autom?ticamente.

In [ ]:
import psi_temporal as psi
import matplotlib.pyplot as plt
from IPython.display import display

REFERENCE_MODE = _central['psi']['REFERENCE_MODE']
REFERENCE_N_MONTHS = _central['psi']['REFERENCE_N_MONTHS']
REFERENCE_MIN_N = _central['psi']['REFERENCE_MIN_N']
MIN_MONTH_OBS = _central['psi']['MIN_MONTH_OBS']
MIN_EXPECTED_BIN_COUNT = _central['psi']['MIN_EXPECTED_BIN_COUNT']
MIN_TRAIN_MONTH_COVERAGE = _central['psi']['MIN_TRAIN_MONTH_COVERAGE']
ALPHA = _central['psi']['ALPHA']
AUTO_DROP_PSI = _central['psi']['AUTO_DROP_PSI']

# Archivo: psi_temporal.py | Funci?n: configure()
# Valida todos los par?metros antes de actualizar la configuraci?n del m?dulo.
configuracion_psi = psi.configure(
    reference_mode=REFERENCE_MODE,
    reference_n_months=REFERENCE_N_MONTHS,
    reference_min_n=REFERENCE_MIN_N,
    min_month_obs=MIN_MONTH_OBS,
    min_expected_bin_count=MIN_EXPECTED_BIN_COUNT,
    min_train_month_coverage=MIN_TRAIN_MONTH_COVERAGE,
    alpha=ALPHA,
    auto_drop_psi=AUTO_DROP_PSI,
)
display(pd.DataFrame(configuracion_psi.items(), columns=["Parametro", "Valor"]))

## PSI.2 ? Validaci?n de muestras y periodos

**Objetivo:** Comprobar ?ndices, valores y periodos sin modificar las muestras.

**Procedencia:** `psi_temporal.py`; el notebook organiza las llamadas y muestra sus resultados.

**Resultado esperado:** Registros y variables por segmento, periodo inicial/final y volumen mensual.



In [ ]:
matrices_psi = {"TRAIN": X_train_v3, "TEST": X_test_v3, "OOT": X_oot_v3}

# Archivo: psi_temporal.py | Funci?n: validate_samples()
# Compara ?ndices, segmentos y valores contra df; convierte periodos sin reset_index.
df_muestras_psi, df_volumen_mensual_psi = psi.validate_samples(df, matrices_psi)
display(df_muestras_psi)
display(df_volumen_mensual_psi.sort_values(["Segmento", "Periodo"]))

## PSI.3 ? C?lculo del PSI temporal

**Objetivo:** Calcular una vez desde las muestras validadas, sin gr?ficos ni escritura.

**Procedencia:** `psi_temporal.py`; el notebook organiza las llamadas y muestra sus resultados.

**Resultado esperado:** Tres dataframes en memoria y confirmaci?n breve.



In [ ]:
# Archivo: psi_temporal.py | Funci?n: run_analysis()
# Internamente llama learn_bins(), transform(), calculate_psi() y classify().
# export=False conserva tablas y artefactos de auditor?a en memoria para PSI.12.
df_psi_mensual, df_resumen_psi, df_referencias_psi = psi.run_analysis(
    df, matrices_psi, reference_mode=REFERENCE_MODE, export=False
)
print(f"PSI calculado: {len(df_resumen_psi)} variables; "
      f"{len(df_psi_mensual)} evaluaciones mensuales. Sin exportaciones.")

## PSI.4 ? Referencia temporal

**Objetivo:** Comprender el origen de la referencia fija por variable.

**Procedencia:** `psi_temporal.py`; el notebook organiza las llamadas y muestra sus resultados.

**Resultado esperado:** Periodos, observaciones, clientes, nulos, m?todo, bins efectivos y estado.

Los meses incluidos en la referencia **no constituyen evaluaciones temporalmente independientes**. Ning?n registro TEST/OOT participa en su aprendizaje.

In [ ]:
# Resultado de run_analysis(), procedente de psi_temporal.py.
# N_bins cuenta bins efectivamente observados, incluida ausencia si existe.
display(df_referencias_psi[[
    "Variable", "Referencia_utilizada", "N_observaciones", "N_clientes",
    "Pct_nulos", "Metodo_binning", "N_bins", "Estado"
]])

## PSI.5 ? Resultado PSI mensual

**Objetivo:** Revisar el detalle ordenado y comparar segmentos mes a mes.

**Procedencia:** `psi_temporal.py`; el notebook organiza las llamadas y muestra sus resultados.

**Resultado esperado:** Tabla completa y tabla con columnas TRAIN, TEST y OOT; ausencias como NaN.



In [ ]:
# Tablas calculadas por run_analysis() en psi_temporal.py; no se recalcula ni exporta.
display(df_psi_mensual.sort_values(["Variable", "Segmento", "Periodo"]))
df_psi_comparacion_mensual = df_psi_mensual.pivot(
    index=["Variable", "Periodo"], columns="Segmento", values="PSI"
).reindex(columns=["TRAIN", "TEST", "OOT"]).sort_index()
display(df_psi_comparacion_mensual)

## PSI.6 ? Clasificaci?n de variables

**Objetivo:** Revisar estados exclusivos basados en TRAIN sin eliminar variables.

**Procedencia:** `psi_temporal.py`; el notebook organiza las llamadas y muestra sus resultados.

**Resultado esperado:** Conteo de los cinco estados y detalle de PSI, rachas, cobertura y decisi?n.

`AUTO_DROP_PSI = False`: ni CANDIDATA_DESCARTE ni NO_EVALUABLE implican exclusi?n autom?tica. NO_EVALUABLE permanece identificado como pendiente de revisi?n.

In [ ]:
# Resultado de classify(), coordinada por run_analysis() en psi_temporal.py.
df_reporte_psi = df_resumen_psi.copy()
variables_candidatas_descarte_psi = df_resumen_psi.loc[df_resumen_psi.Candidata_descarte, "Variable"].tolist()
variables_revision_psi = df_resumen_psi.loc[df_resumen_psi.Estado_PSI != "APROBADA", "Variable"].tolist()

df_conteo_estados_psi = df_resumen_psi.Estado_PSI.value_counts().reindex(
    psi.PSI_STATES, fill_value=0
).rename_axis("Estado_PSI").reset_index(name="Variables")
display(df_conteo_estados_psi)
display(df_resumen_psi)

df_decisiones_psi = df_resumen_psi[["Variable", "PSI_max_train", "Meses_train_rojos","Max_meses_rojos_consecutivos", "Cobertura_temporal", "Estado_PSI"]].copy()

df_decisiones_psi["Decision_final"] = df_decisiones_psi.Estado_PSI.map(lambda estado: "Conservar" if estado == "APROBADA" else "Conservar para revisión")
df_decisiones_psi

## PSI.7 ? Gr?fico individual opcional

**Objetivo:** Elegir una sola variable para revisar su evoluci?n.

**Procedencia:** `psi_temporal.py`; el notebook organiza las llamadas y muestra sus resultados.

**Resultado esperado:** Un gr?fico seleccionado; dropdown si ipywidgets est? disponible.



In [ ]:
# Cambie este nombre y reejecute solo esta celda para seleccionar sin widgets.
VARIABLE_PSI = df_resumen_psi.Variable.iloc[0]
USAR_WIDGETS_PSI = True
# El ejecutor autom?tico inyecta True para usar una selecci?n reproducible.
try:
    PSI_EJECUCION_AUTOMATICA
except NameError:
    PSI_EJECUCION_AUTOMATICA = False


def mostrar_variable_psi(variable):
    # Archivo: psi_temporal.py | Funci?n: plot_psi_variable()
    # Construye desde los dataframes en memoria, sin leer ni escribir PNG.
    figura, _ = psi.plot_psi_variable(variable, df_psi_mensual, df_resumen_psi)
    figura.tight_layout()
    plt.show()
    plt.close(figura)


try:
    if not USAR_WIDGETS_PSI or PSI_EJECUCION_AUTOMATICA:
        raise ImportError("Selecci?n simple activada")
    import ipywidgets as widgets
except ImportError:
    mostrar_variable_psi(VARIABLE_PSI)
else:
    selector_variable_psi = widgets.Dropdown(
        options=df_resumen_psi.Variable.tolist(), value=VARIABLE_PSI,
        description="Variable:", layout=widgets.Layout(width="95%")
    )
    vista_individual_psi = widgets.interactive_output(
        mostrar_variable_psi, {"variable": selector_variable_psi}
    )
    display(selector_variable_psi, vista_individual_psi)

## PSI.8 ? Dashboard principal de subplots

**Objetivo:** Comparar variables en p?ginas legibles de hasta 12 gr?ficos.

**Procedencia:** `psi_temporal.py`; el notebook organiza las llamadas y muestra sus resultados.

**Resultado esperado:** P?ginas visibles directamente en el notebook, cuatro columnas y una leyenda general.



In [ ]:
# Opciones: TODAS, APROBADA, EN_OBSERVACION, REVISION_PRIORITARIA,
# CANDIDATA_DESCARTE y NO_EVALUABLE. Cambiar el filtro no recalcula PSI.
ESTADO_DASHBOARD_PSI = "TODAS"
COLUMNAS_DASHBOARD_PSI = 4
VARIABLES_POR_PAGINA_PSI = 12

# Archivo: psi_temporal.py | Función: plot_psi_dashboard()
# Devuelve figuras de plt.subplots() creadas con datos en memoria, sin exportaci?n.
figuras_dashboard_psi = psi.plot_psi_dashboard(
    df_psi_mensual, df_resumen_psi, estado=ESTADO_DASHBOARD_PSI,
    columnas=COLUMNAS_DASHBOARD_PSI, por_pagina=VARIABLES_POR_PAGINA_PSI
)
if figuras_dashboard_psi:
    plt.show()  # Muestra todas las p?ginas abiertas; no lee PNG guardados.
    for figura in figuras_dashboard_psi:
        plt.close(figura)
else:
    print(f"No hay variables con estado {ESTADO_DASHBOARD_PSI}.")

## PSI.9 ? Heatmaps por segmento

**Objetivo:** Ver el patr?n temporal de TRAIN, TEST y OOT por separado.

**Procedencia:** `psi_temporal.py`; el notebook organiza las llamadas y muestra sus resultados.

**Resultado esperado:** Tres heatmaps en memoria; gris indica ausencia de evaluaci?n.



In [ ]:
for segmento in ["TRAIN", "TEST", "OOT"]:
    # Archivo: psi_temporal.py | Funci?n: plot_psi_heatmap()
    # Mantiene periodos ausentes enmascarados, diferentes de PSI=0.
    figura, _ = psi.plot_psi_heatmap(df_psi_mensual, segmento)
    plt.show()
    plt.close(figura)

## PSI.10 ? Reporte consolidado

**Objetivo:** Consolidar filtros anteriores y PSI de forma independiente de los gr?ficos.

**Procedencia:** `psi_temporal.py`; el notebook organiza las llamadas y muestra sus resultados.

**Resultado esperado:** Reporte por variable/filtro sin duplicados; ninguna escritura.



In [ ]:
# Archivo: psi_temporal.py | Funci?n: consolidate()
# Reconstruye desde las listas originales: reejecutar no acumula registros.
df_reporte_univariado = psi.consolidate(
    X_train, variables_descartadas, variables_descartadas_f3, df_resumen_psi
)
assert not df_reporte_univariado.duplicated(["Variable", "Filtro"]).any()
display(df_reporte_univariado)

## PSI.11 ? Construcci?n de matrices finales

**Objetivo:** Conservar el contrato v3/v4 y la correspondencia con target, periodo y segmento.

**Procedencia:** `psi_temporal.py`; el notebook organiza las llamadas y muestra sus resultados.

**Resultado esperado:** Dimensiones antes/despu?s y validaciones de integridad.



In [ ]:
EXCLUSIONES_PSI_DOCUMENTADAS = {}  # variable: justificaci?n expl?cita
if AUTO_DROP_PSI:
    if not EXCLUSIONES_PSI_DOCUMENTADAS or not all(
        isinstance(motivo, str) and motivo.strip()
        for motivo in EXCLUSIONES_PSI_DOCUMENTADAS.values()
    ):
        raise ValueError("Documente una justificaci?n no vac?a por cada exclusi?n")
variables_descartar_psi = list(EXCLUSIONES_PSI_DOCUMENTADAS) if AUTO_DROP_PSI else []

X_train_v4 = X_train_v3.drop(columns=variables_descartar_psi).copy()
X_test_v4 = X_test_v3.drop(columns=variables_descartar_psi).copy()
X_oot_v4 = X_oot_v3.drop(columns=variables_descartar_psi).copy()

matrices_finales_psi = {"TRAIN": X_train_v4, "TEST": X_test_v4, "OOT": X_oot_v4}
objetivos_psi = {"TRAIN": y_train, "TEST": y_test, "OOT": y_oot}
for segmento, matriz in matrices_finales_psi.items():
    original = matrices_psi[segmento]
    pd.testing.assert_frame_equal(matriz, original.drop(columns=variables_descartar_psi))
    pd.testing.assert_series_equal(
        objetivos_psi[segmento], df.loc[matriz.index, "indicador_default"]
    )
    assert matriz.index.equals(original.index)
    assert df.loc[matriz.index, "SEGMENTO"].eq(segmento).all()
    assert "periodo_observacion" not in matriz.columns

# Archivo: psi_temporal.py | Funci?n: validate_samples()
# Revalida valores, ?ndices y periodos contra df para las matrices finales.
psi.validate_samples(df, matrices_finales_psi)
display(pd.DataFrame([
    {"Segmento": segmento, "Filas_v3": len(matrices_psi[segmento]),
     "Filas_v4": len(matriz), "Variables_v3": matrices_psi[segmento].shape[1],
     "Variables_v4": matriz.shape[1]}
    for segmento, matriz in matrices_finales_psi.items()
]))
print("?ndices, registros, target, periodos y segmentos validados.")

## PSI.12 ? Exportaci?n expl?cita de resultados

**Objetivo:** Guardar ?nicamente los resultados ya calculados cuando ejecute esta celda.

**Procedencia:** `psi_temporal.py`; el notebook organiza las llamadas y muestra sus resultados.

**Resultado esperado:** CSV, auditor?a, dashboards y heatmaps; individuales solo si los solicita.



In [ ]:
# Puede omitir por completo esta celda durante la exploraci?n interactiva.
EXPORTAR_INDIVIDUALES_PSI = False
EXPORTAR_DASHBOARDS_PSI = True
EXPORTAR_HEATMAPS_PSI = True
ESTADOS_EXPORTAR_PSI = ("TODAS",)  # Puede a?adir otros estados de PSI.6.

# Archivo: psi_temporal.py | Funci?n: export_psi_tables()
# Escribe tablas y artefactos aprendidos en PSI.3; no vuelve a calcular PSI.
archivos_tablas_psi = psi.export_psi_tables(df_psi_mensual, df_resumen_psi, df_referencias_psi, df_reporte_univariado)
# Archivo: psi_temporal.py | Funci?n: export_psi_figures()
# Dibuja y guarda solo los tipos solicitados; conserva copia de salidas existentes.
archivos_figuras_psi = psi.export_psi_figures(
    df_psi_mensual, df_resumen_psi,
    individuales=EXPORTAR_INDIVIDUALES_PSI,
    dashboards=EXPORTAR_DASHBOARDS_PSI,
    heatmaps=EXPORTAR_HEATMAPS_PSI,
    estados=ESTADOS_EXPORTAR_PSI,
    columnas=COLUMNAS_DASHBOARD_PSI,
    por_pagina=VARIABLES_POR_PAGINA_PSI,
)
print(f"Exportadas {len(archivos_tablas_psi)} tablas/artefactos y "
      f"{len(archivos_figuras_psi)} figuras en {psi.ROOT}.")

In [ ]:
X_train_v4.columns

In [ ]:
# Puente de persistencia para ejecución manual del notebook; no recalcula estadísticas.
from pipeline_runtime import persist_psi_contract
manifest_v4 = persist_psi_contract(globals())
print('Contrato v4 disponible para notebook 3:', manifest_v4)